# LLM Playlist Recommender — End-to-End Colab Pipeline

This notebook runs the full pipeline described in the repository sequentially:

1. Mount Google Drive and configure paths
2. Install dependencies
3. Clone the repository
4. Convert MPD JSON → CSV
5. Compute track-based playlist embeddings (pre-trained SentenceBERT)
6. K-means clustering
7. Add exact-match percentages
8. Clean miscellaneous clusters
9. Train/val/test split
10. Fine-tune SentenceBERT (cross-entropy or triplet loss)
11. Generate final playlist-title embeddings with fine-tuned model
12. Run inference

**Prerequisites:**
- Upload the [Spotify Million Playlist Dataset](https://www.aicrowd.com/challenges/spotify-million-playlist-dataset-challenge) JSON slices to your Google Drive before starting.
- A GPU runtime is strongly recommended (Runtime > Change runtime type > T4 GPU for steps 1–11; A100 required for the LLM reranking step).


## Step 1 — Mount Google Drive and configure paths

Edit the path variables below to match where you stored the MPD dataset on your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Edit these paths to match your Drive layout ──────────────────────────────
DRIVE_BASE        = "/content/drive/MyDrive/playlist-recommender"

MPD_JSON_DIR      = f"{DRIVE_BASE}/million_playlist_dataset"   # MPD .json slices
CSVS_DIR          = f"{DRIVE_BASE}/csvs"                       # output CSVs
EMBEDDINGS_DIR    = f"{DRIVE_BASE}/embeddings"                 # raw embeddings pkl
CLUSTERS_DIR      = f"{DRIVE_BASE}/clusters"                   # clustering output
CLEAN_DIR         = f"{DRIVE_BASE}/clean"                      # cleaned clusters
SPLIT_DIR         = f"{DRIVE_BASE}/split"                      # train/val/test CSVs
MODEL_CE_DIR      = f"{DRIVE_BASE}/model_cross_entropy"        # cross-entropy model
MODEL_TL_DIR      = f"{DRIVE_BASE}/model_triplet_loss"         # triplet-loss model
FINAL_EMBEDS_DIR  = f"{DRIVE_BASE}/final_embeddings"           # final title embeds
REPO_DIR          = "/content/LLM-Playlist-Recommender"        # cloned repo

import os
for d in [CSVS_DIR, EMBEDDINGS_DIR, CLUSTERS_DIR, CLEAN_DIR,
          SPLIT_DIR, MODEL_CE_DIR, MODEL_TL_DIR, FINAL_EMBEDS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Paths configured.")

## Step 2 — Install dependencies

In [ ]:
%pip install -q \
    sentence-transformers \
    transformers \
    datasets \
    evaluate \
    scikit-learn \
    gensim \
    langchain-core \
    openai \
    tqdm \
    pyyaml \
    pandas \
    numpy

print("Dependencies installed.")

## Step 3 — Clone the repository and add it to the Python path

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/elea-vellard/LLM-Playlist-Recommender.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)

## Step 4 — Convert MPD JSON slices → CSV files

Reads all `.json` slices from `MPD_JSON_DIR` and writes `items.csv`, `playlists.csv`, `tracks.csv`, and `playlists_descr.csv` to `CSVS_DIR`.

In [ ]:
import os, importlib.util

spec = importlib.util.spec_from_file_location(
    "json2csv",
    os.path.join(REPO_DIR, "transform-dataset", "json2csv.py"),
)
json2csv_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(json2csv_mod)

json2csv_mod.convert(input_dir=MPD_JSON_DIR, output_dir=CSVS_DIR)
print("CSV files written to:", CSVS_DIR)

## Step 5 — Compute track-based playlist embeddings

Each playlist is represented as the mean of its track-title embeddings using the pre-trained `all-mpnet-base-v2` SentenceBERT model. Result is saved as `embeddings.pkl`.

In [ ]:
import os, importlib.util

spec = importlib.util.spec_from_file_location(
    "track_embeddings",
    os.path.join(REPO_DIR, "clustering-no-split", "embeddings", "track_embeddings_no-split.py"),
)
track_emb_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(track_emb_mod)

EMBEDDINGS_PKL = os.path.join(EMBEDDINGS_DIR, "embeddings.pkl")

track_emb_mod.run(
    playlists_csv = os.path.join(CSVS_DIR, "playlists.csv"),
    items_csv     = os.path.join(CSVS_DIR, "items.csv"),
    tracks_csv    = os.path.join(CSVS_DIR, "tracks.csv"),
    output_file   = EMBEDDINGS_PKL,
    model_name    = "sentence-transformers/all-mpnet-base-v2",
)
print("Embeddings saved to:", EMBEDDINGS_PKL)

## Step 6 — K-means clustering (200 clusters)

In [ ]:
import os
# Note: the file is named with a hyphen so we import via importlib
import importlib.util, sys

spec = importlib.util.spec_from_file_location(
    "clustering_no_split",
    os.path.join(REPO_DIR, "clustering-no-split/clusters/clustering-no-split.py")
)
clustering_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(clustering_mod)

CLUSTERS_CSV = os.path.join(CLUSTERS_DIR, "clusters.csv")

clustering_mod.run(
    embeddings_file = EMBEDDINGS_PKL,
    output_file     = CLUSTERS_CSV,
    num_clusters    = 200,
)
print("Clusters written to:", CLUSTERS_CSV)

## Step 7 — Add exact-match percentage column to clusters CSV

In [ ]:
import importlib.util, os

spec = importlib.util.spec_from_file_location(
    "percent_no_split",
    os.path.join(REPO_DIR, "clustering-no-split/clusters/percent-no-split.py")
)
percent_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(percent_mod)

CLUSTERS_PCT_CSV = os.path.join(CLUSTERS_DIR, "clusters_with_exact_matches.csv")

percent_mod.analyze_clusters_with_exact_matches(
    input_file  = CLUSTERS_CSV,
    output_file = CLUSTERS_PCT_CSV,
)
print("Annotated clusters written to:", CLUSTERS_PCT_CSV)

## Step 8 — Clean miscellaneous clusters

Removes clusters whose most-frequent playlist title appears in ≤ 2% of the cluster's rows.

In [ ]:
import importlib.util, os

spec = importlib.util.spec_from_file_location(
    "clean_clusters",
    os.path.join(REPO_DIR, "clustering-no-split/clean/clean-clusters-no-split.py")
)
clean_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(clean_mod)

CLEAN_CSV = os.path.join(CLEAN_DIR, "clusters_with_exact_matches.csv")

clean_mod.clean_clusters(
    input_file  = CLUSTERS_PCT_CSV,
    output_file = CLEAN_CSV,
    threshold   = 2.0,
)
print("Cleaned clusters written to:", CLEAN_CSV)

## Step 9 — Split clusters into train / val / test sets (80 / 10 / 10)

In [ ]:
import os, importlib.util

spec = importlib.util.spec_from_file_location(
    "split_represented",
    os.path.join(REPO_DIR, "clustering-no-split", "split", "split_represented.py"),
)
split_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(split_mod)

split_mod.split_clusters(
    input_clusters_file = CLEAN_CSV,
    output_dir          = SPLIT_DIR,
    seed                = 1,
    val_ratio           = 0.1,
    test_ratio          = 0.1,
)
print("Split CSVs written to:", SPLIT_DIR)

## Step 10 — Fine-tune SentenceBERT on cluster labels

Run **one** of the two cells below. The cross-entropy model (`MODEL_CE_DIR`) tends to be used by the rest of the pipeline.

> ⚠️ These cells can take a long time. Reduce `epochs` for a quick test run.

In [ ]:
# Option A: Cross-entropy fine-tuning (used by the rest of the pipeline)
import os, importlib.util

spec = importlib.util.spec_from_file_location(
    "cross_entropy_model_finetuning",
    os.path.join(REPO_DIR, "finetuning", "cross_entropy_model_finetuning.py"),
)
ce_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ce_mod)

ce_mod.run(
    train_csv     = os.path.join(SPLIT_DIR, "clusters_train.csv"),
    val_csv       = os.path.join(SPLIT_DIR, "clusters_val.csv"),
    output_dir    = MODEL_CE_DIR,
    model_name    = "sentence-transformers/all-MiniLM-L6-v2",
    batch_size    = 8,
    epochs        = 5,        # increase to 100 for full training
    learning_rate = 2e-5,
    warmup_steps  = 100,
)
print("Cross-entropy model saved to:", MODEL_CE_DIR)

In [ ]:
# Option B: Triplet-loss fine-tuning (alternative — skip if using Option A)
import os, importlib.util

spec = importlib.util.spec_from_file_location(
    "finetuning_triplet_loss",
    os.path.join(REPO_DIR, "finetuning", "finetuning_triplet_loss.py"),
)
tl_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tl_mod)

tl_mod.run(
    train_csv     = os.path.join(SPLIT_DIR, "clusters_train.csv"),
    val_csv       = os.path.join(SPLIT_DIR, "clusters_val.csv"),
    output_dir    = MODEL_TL_DIR,
    model_name    = "sentence-transformers/all-MiniLM-L6-v2",
    batch_size    = 8,
    epochs        = 5,        # increase to 50 for full training
    learning_rate = 2e-5,
)
print("Triplet-loss model saved to:", MODEL_TL_DIR)

## Step 11 — Generate final playlist-title embeddings with the fine-tuned model

Embeds every playlist title from `playlists.csv` using the fine-tuned model and saves to `playlists_embeddings.pkl`. This is the file used at inference time.

In [ ]:
import os, importlib.util

spec = importlib.util.spec_from_file_location(
    "playlists_embeddings_final",
    os.path.join(REPO_DIR, "embeddings", "playlists_embeddings_final.py"),
)
embeds_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embeds_mod)

FINAL_EMBEDS_PKL = os.path.join(FINAL_EMBEDS_DIR, "playlists_embeddings.pkl")

embeds_mod.run(
    playlists_csv       = os.path.join(CSVS_DIR, "playlists.csv"),
    output_file         = FINAL_EMBEDS_PKL,
    finetuned_model_dir = MODEL_CE_DIR,   # swap to MODEL_TL_DIR to use triplet-loss model
)
print("Final embeddings saved to:", FINAL_EMBEDS_PKL)

## Step 12 — Inference: generate recommendations for a playlist title

**Option A** uses the embedding similarity search only (no LLM reranking) — runs on a free T4 GPU.

**Option B** adds the LLM reranking step — requires an A100 (Colab Pro+) due to the 7B parameter model.

In [ ]:
# Option A: Similarity-based recommendations only (no LLM, runs on T4)
import os, importlib.util

spec = importlib.util.spec_from_file_location(
    "recommend",
    os.path.join(REPO_DIR, "similarity", "recommend.py"),
)
rec_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rec_mod)

PLAYLIST_NAME = "chill vibes"   # ← change this to your query

tokenizer, model = rec_mod.load_fine_tuned_model(MODEL_CE_DIR)
playlist_embeddings = rec_mod.load_playlist_embeddings(FINAL_EMBEDS_PKL)
playlist_tracks = rec_mod.load_playlist_tracks_with_artists(
    os.path.join(CSVS_DIR, "items.csv"),
    os.path.join(CSVS_DIR, "tracks.csv"),
)

similar = rec_mod.find_similar_playlists(PLAYLIST_NAME, playlist_embeddings, tokenizer, model, top_k=50)
top_songs = rec_mod.get_top_songs_with_artists(similar, playlist_tracks, top_k=10)

print(f"\nTop 10 recommendations for '{PLAYLIST_NAME}':")
for i, ((song, artist), count) in enumerate(top_songs, 1):
    print(f"  {i:2}. {song} — {artist}  (votes: {count})")

In [ ]:
# Option B: Full pipeline with LLM reranking (requires A100 — Colab Pro+)
import os, importlib.util

spec = importlib.util.spec_from_file_location(
    "LLM_integrated",
    os.path.join(REPO_DIR, "LLM_part", "LLM_integrated"),
)
llm_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(llm_mod)

PLAYLIST_NAME = "chill vibes"   # ← change this to your query

# Load fine-tuned embedder + LLM
tokenizer, model = llm_mod.load_fine_tuned_model(MODEL_CE_DIR)
embeds = llm_mod.load_playlist_embeddings(FINAL_EMBEDS_PKL)
tracks = llm_mod.load_playlist_tracks_with_artists(
    os.path.join(CSVS_DIR, "items.csv"),
    os.path.join(CSVS_DIR, "tracks.csv"),
)
titles = llm_mod.load_playlist_titles(os.path.join(CSVS_DIR, "playlists.csv"))
kv = llm_mod.load_embeddings_to_keyedvectors(embeds)

recs = llm_mod.recommend(
    PLAYLIST_NAME,
    kv,
    tokenizer,
    model,
    tracks,
    titles,
    top_k_similar=50,
    top_n_recs=10,
    llm_model_key="togethercomputer/LLaMA-2-7B-32K",
)

print(f"\nLLM-ranked Top 10 recommendations for '{PLAYLIST_NAME}':")
for i, r in enumerate(recs, 1):
    print(f"  {i:2}. {r['song']} — {r['artist']}")